# Phase 4 — Agent Training

Run every cell top to bottom. This notebook:
1. Trains a DQN agent and a PPO agent
2. Evaluates both on the validation set (2020-2021)
3. Plots reward curves, trade signals, and portfolio comparison
4. Prints a DQN vs PPO vs Buy & Hold comparison table

**Expected runtime:** ~5-15 minutes depending on your machine (CPU is fine for 200k steps).

---
## Cell 1 — Imports & path setup

In [3]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

from env.trading_env import TradingEnv
import importlib
import data.pipeline as pipeline_mod
importlib.reload(pipeline_mod)
run_pipeline = pipeline_mod.run_pipeline

TICKER      = 'AAPL'
WINDOW_SIZE = 10
MODEL_DIR   = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('[PASS] All imports successful.')
print(f'       Models will be saved to: {MODEL_DIR}')

ImportError: cannot import name 'run_pipeline' from 'data.pipeline' (/Users/abdulmoiz/Downloads/RL Trading Agent/data/pipeline.py)

---
## Cell 2 — Load data (scaled features + raw prices)

In [ ]:
train_df, val_df, test_df = run_pipeline(TICKER)

raw_all = pd.read_csv(
    os.path.join(PROJECT_ROOT, 'data', 'raw', f'{TICKER}.csv'),
    index_col=0, parse_dates=True,
)
if isinstance(raw_all.columns, pd.MultiIndex):
    raw_all.columns = raw_all.columns.get_level_values(0)

train_raw = raw_all.loc[train_df.index]
val_raw   = raw_all.loc[val_df.index]
test_raw  = raw_all.loc[test_df.index]

print(f'Train : {len(train_df)} rows | Val : {len(val_df)} rows | Test : {len(test_df)} rows')
assert (train_raw['Close'] > 0).all(), 'FAIL: Negative prices in train_raw'
print('[PASS] Data loaded. Raw prices confirmed positive.')

---
## Cell 3 — Build environments

In [ ]:
def make_env(df, raw_df):
    def _init():
        env = TradingEnv(df, raw_df, window_size=WINDOW_SIZE)
        return Monitor(env)
    return _init

train_env = DummyVecEnv([make_env(train_df, train_raw)])
val_env   = DummyVecEnv([make_env(val_df,   val_raw)])

print(f'Train env obs shape : {train_env.observation_space.shape}')
print(f'Action space        : {train_env.action_space}')
print('[PASS] Environments built and wrapped in DummyVecEnv.')

---
## Cell 4 — Train DQN

This will print live progress. Watch for the reward trending upward over episodes.
Training ~200k steps takes roughly 3-8 minutes on CPU.

In [ ]:
# EvalCallback saves the best checkpoint automatically
stop_cb = StopTrainingOnNoModelImprovement(
    max_no_improvement_evals=10, min_evals=20, verbose=1
)
eval_cb_dqn = EvalCallback(
    val_env,
    best_model_save_path = os.path.join(MODEL_DIR, 'dqn_best'),
    log_path             = os.path.join(MODEL_DIR, 'dqn_logs'),
    eval_freq            = 5_000,
    n_eval_episodes      = 3,
    deterministic        = True,
    callback_after_eval  = stop_cb,
    verbose              = 1,
)

dqn_model = DQN(
    policy                  = 'MlpPolicy',
    env                     = train_env,
    learning_rate           = 1e-4,
    buffer_size             = 50_000,
    learning_starts         = 1_000,
    batch_size              = 64,
    gamma                   = 0.99,
    train_freq              = 4,
    target_update_interval  = 500,
    exploration_fraction    = 0.15,
    exploration_final_eps   = 0.05,
    verbose                 = 1,
)

print('Training DQN for 200,000 steps...')
dqn_model.learn(
    total_timesteps = 200_000,
    callback        = eval_cb_dqn,
    progress_bar    = True,
)

dqn_save = os.path.join(MODEL_DIR, f'dqn_{TICKER.lower()}')
dqn_model.save(dqn_save)
print(f'\n[PASS] DQN trained and saved → {dqn_save}.zip')

---
## Cell 5 — Train PPO

In [ ]:
# Rebuild train_env — DQN training exhausted the old one
train_env = DummyVecEnv([make_env(train_df, train_raw)])

stop_cb2 = StopTrainingOnNoModelImprovement(
    max_no_improvement_evals=10, min_evals=20, verbose=1
)
eval_cb_ppo = EvalCallback(
    val_env,
    best_model_save_path = os.path.join(MODEL_DIR, 'ppo_best'),
    log_path             = os.path.join(MODEL_DIR, 'ppo_logs'),
    eval_freq            = 5_000,
    n_eval_episodes      = 3,
    deterministic        = True,
    callback_after_eval  = stop_cb2,
    verbose              = 1,
)

ppo_model = PPO(
    policy        = 'MlpPolicy',
    env           = train_env,
    learning_rate = 3e-4,
    n_steps       = 2048,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.01,
    vf_coef       = 0.5,
    max_grad_norm = 0.5,
    verbose       = 1,
)

print('Training PPO for 200,000 steps...')
ppo_model.learn(
    total_timesteps = 200_000,
    callback        = eval_cb_ppo,
    progress_bar    = True,
)

ppo_save = os.path.join(MODEL_DIR, f'ppo_{TICKER.lower()}')
ppo_model.save(ppo_save)
print(f'\n[PASS] PPO trained and saved → {ppo_save}.zip')

---
## Cell 6 — Evaluate both agents on the validation set

In [ ]:
def run_episode(model, df, raw_df, window_size=10, initial_balance=10_000):
    """Run one full deterministic episode. Returns env after completion."""
    env = TradingEnv(df, raw_df, initial_balance=initial_balance,
                     window_size=window_size)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = env.step(int(action))
        done = terminated or truncated
    return env

def compute_metrics(env, initial_balance=10_000):
    portfolio = pd.Series(env.portfolio_history)
    returns   = portfolio.pct_change().dropna()
    final     = portfolio.iloc[-1]
    ret_pct   = (final - initial_balance) / initial_balance * 100
    sharpe    = (returns.mean() / (returns.std() + 1e-8)) * np.sqrt(252)
    dd        = ((portfolio - portfolio.cummax()) / portfolio.cummax()).min() * 100
    tlog      = env.get_trade_log()
    n_trades  = len(tlog)
    if n_trades > 0 and 'action' in tlog.columns:
        sells    = tlog[tlog['action']=='SELL']
        avg_buy  = tlog[tlog['action']=='BUY']['price'].mean() if len(tlog[tlog['action']=='BUY']) > 0 else 0
        win_rate = (sells['price'] > avg_buy).mean() * 100 if len(sells) > 0 else 0
    else:
        win_rate = 0
    return {
        'final_value'  : round(final, 2),
        'total_return' : round(ret_pct, 2),
        'sharpe'       : round(float(sharpe), 3),
        'max_drawdown' : round(float(dd), 2),
        'n_trades'     : n_trades,
        'win_rate'     : round(win_rate, 1),
    }

print('Running DQN on validation set...')
dqn_env = run_episode(dqn_model, val_df, val_raw)
dqn_metrics = compute_metrics(dqn_env)

print('Running PPO on validation set...')
ppo_env = run_episode(ppo_model, val_df, val_raw)
ppo_metrics = compute_metrics(ppo_env)

# Buy & Hold baseline
bnh_start = val_raw['Close'].iloc[0]
bnh_end   = val_raw['Close'].iloc[-1]
bnh_return = (bnh_end / bnh_start - 1) * 100
bnh_series = 10_000 * (val_raw['Close'] / bnh_start)
bnh_returns = bnh_series.pct_change().dropna()
bnh_sharpe  = (bnh_returns.mean() / (bnh_returns.std() + 1e-8)) * np.sqrt(252)
bnh_dd      = ((bnh_series - bnh_series.cummax()) / bnh_series.cummax()).min() * 100

print('\n' + '='*62)
print(f"  {'Metric':<25} {'DQN':>10} {'PPO':>10} {'Buy&Hold':>10}")
print('='*62)
rows = [
    ('Final portfolio ($)', 'final_value',  f"{10_000*(1+bnh_return/100):,.0f}"),
    ('Total return (%)',    'total_return', f"{bnh_return:+.2f}"),
    ('Sharpe ratio',        'sharpe',       f"{bnh_sharpe:.3f}"),
    ('Max drawdown (%)',    'max_drawdown', f"{bnh_dd:.2f}"),
    ('Trades',              'n_trades',     'N/A'),
    ('Win rate (%)',        'win_rate',     'N/A'),
]
for label, key, bnh_val in rows:
    print(f"  {label:<25} {str(dqn_metrics[key]):>10} {str(ppo_metrics[key]):>10} {bnh_val:>10}")
print('='*62)

---
## Cell 7 — Portfolio value comparison chart

In [ ]:
dqn_portfolio = pd.Series(dqn_env.portfolio_history)
ppo_portfolio = pd.Series(ppo_env.portfolio_history)
bnh_vals      = bnh_series.values[:max(len(dqn_portfolio), len(ppo_portfolio))]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Portfolio value
axes[0].plot(dqn_portfolio.values, label='DQN',        color='steelblue', linewidth=1.2)
axes[0].plot(ppo_portfolio.values, label='PPO',        color='darkorange', linewidth=1.2)
axes[0].plot(bnh_vals[:len(dqn_portfolio)],
             label='Buy & Hold', color='gray', linewidth=1, linestyle='--')
axes[0].axhline(10_000, color='black', linestyle=':', linewidth=0.8, alpha=0.5)
axes[0].set_title('Portfolio value — DQN vs PPO vs Buy & Hold (validation set 2020–2021)')
axes[0].set_ylabel('Portfolio value ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Price chart with DQN trade signals
price_vals = val_raw['Close'].values
axes[1].plot(price_vals, color='steelblue', linewidth=0.8, label='AAPL close')

for env_obj, color_b, color_s, label in [
    (dqn_env, 'lime',   'red',    'DQN'),
    (ppo_env, 'green',  'tomato', 'PPO'),
]:
    tlog = env_obj.get_trade_log()
    if len(tlog) > 0:
        buys  = tlog[tlog['action']=='BUY']
        sells = tlog[tlog['action']=='SELL']
        buy_idx  = (buys['step'].values  - WINDOW_SIZE).clip(0, len(price_vals)-1)
        sell_idx = (sells['step'].values - WINDOW_SIZE).clip(0, len(price_vals)-1)
        axes[1].scatter(buy_idx,  price_vals[buy_idx],
                        marker='^', color=color_b, s=35, zorder=5,
                        label=f'{label} Buy ({len(buys)})')
        axes[1].scatter(sell_idx, price_vals[sell_idx],
                        marker='v', color=color_s, s=35, zorder=5,
                        label=f'{label} Sell ({len(sells)})')

axes[1].set_title('Trade signals on price chart')
axes[1].set_ylabel('Price ($)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('[PASS] Portfolio comparison chart rendered.')

---
## Cell 8 — Reward curve during training (from EvalCallback logs)

In [ ]:
def load_eval_log(log_path):
    """Load EvalCallback evaluation log if it exists."""
    fpath = os.path.join(log_path, 'evaluations.npz')
    if not os.path.exists(fpath):
        return None, None
    data      = np.load(fpath)
    timesteps = data['timesteps']
    results   = data['results'].mean(axis=1)   # mean across eval episodes
    return timesteps, results

dqn_steps, dqn_rewards = load_eval_log(os.path.join(MODEL_DIR, 'dqn_logs'))
ppo_steps, ppo_rewards = load_eval_log(os.path.join(MODEL_DIR, 'ppo_logs'))

if dqn_steps is not None or ppo_steps is not None:
    fig, ax = plt.subplots(figsize=(12, 4))
    if dqn_steps is not None:
        ax.plot(dqn_steps, dqn_rewards, label='DQN eval reward',
                color='steelblue', linewidth=1.2)
    if ppo_steps is not None:
        ax.plot(ppo_steps, ppo_rewards, label='PPO eval reward',
                color='darkorange', linewidth=1.2)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title('Mean eval reward during training')
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Mean episode reward')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('[PASS] Reward curves rendered.')
    print('[INFO] Healthy sign: reward trends upward over timesteps.')
else:
    print('[INFO] EvalCallback log files not found — skipping reward curve.')
    print('       This is fine if EvalCallback did not fire yet (< 5000 steps trained).')

---
## Cell 9 — Load saved models & verify (reload test)

In [ ]:
dqn_path = os.path.join(MODEL_DIR, f'dqn_{TICKER.lower()}.zip')
ppo_path = os.path.join(MODEL_DIR, f'ppo_{TICKER.lower()}.zip')

assert os.path.exists(dqn_path), f'FAIL: DQN model not found at {dqn_path}'
assert os.path.exists(ppo_path), f'FAIL: PPO model not found at {ppo_path}'

# Reload from disk — simulates what the dashboard will do in Phase 6
dqn_reloaded = DQN.load(dqn_path)
ppo_reloaded = PPO.load(ppo_path)

# Quick prediction test
dummy_obs = np.zeros((1, WINDOW_SIZE, train_df.shape[1]), dtype=np.float32)
dqn_action, _ = dqn_reloaded.predict(dummy_obs, deterministic=True)
ppo_action, _ = ppo_reloaded.predict(dummy_obs, deterministic=True)

print(f'DQN model reloaded. Dummy obs → action: {dqn_action}')
print(f'PPO model reloaded. Dummy obs → action: {ppo_action}')
assert dqn_action[0] in [0, 1, 2], 'FAIL: DQN action out of range'
assert ppo_action[0] in [0, 1, 2], 'FAIL: PPO action out of range'
print()
print('[PASS] Both models save/load correctly. Ready for Phase 5 backtesting.')
print(f'       DQN: {os.path.getsize(dqn_path)/1024:.1f} KB')
print(f'       PPO: {os.path.getsize(ppo_path)/1024:.1f} KB')

---
## Cell 10 — Phase 4 summary

In [ ]:
print('='*55)
print('  Phase 4 — complete')
print('='*55)
print(f"  DQN model   : models/dqn_{TICKER.lower()}.zip")
print(f"  PPO model   : models/ppo_{TICKER.lower()}.zip")
print(f"  DQN return  : {dqn_metrics['total_return']:+.2f}%")
print(f"  PPO return  : {ppo_metrics['total_return']:+.2f}%")
print(f"  Buy & Hold  : {bnh_return:+.2f}%")
print()
best = 'DQN' if dqn_metrics['sharpe'] > ppo_metrics['sharpe'] else 'PPO'
print(f"  Best Sharpe : {best}")
print()
print('  Next: Phase 5 — backtest on the test set (2022-2024)')
print('        This is the first time the test set is touched.')
print('='*55)